# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library. The dataset focuses on protein abundance, modifications, and sequences in human samples derived from extracellular vesicles, and is defined using a Croissant schema.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# View basic metadata (access values as attributes)
metadata = dataset.metadata
print(f"Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Published: {getattr(metadata, 'datePublished', 'N/A')}")
print(f"Authors: {getattr(metadata, 'author', 'N/A')}")

## 2. Data Overview
Review available *record sets* (tables), their fields (columns), and their unique `@id` references required for downstream data loading and processing.

In [ ]:
# Explore all record sets in the dataset
# Each record set has a unique @id and a set of fields. This step lists them for reference.
record_sets = list(dataset.metadata.recordSet)
for rs in record_sets:
    print(f"Record Set Name: {getattr(rs, 'name', 'N/A')}")
    print(f"  @id: {getattr(rs, '@id', 'N/A')}")
    print(f"  Description: {getattr(rs, 'description', 'N/A')}")
    print("  Fields:")
    for field in getattr(rs, 'field', []):
        print(f"    - Field Name: {getattr(field, 'name', 'N/A')} ( @id: {getattr(field, '@id', 'N/A')} | dataType: {getattr(field, 'dataType', 'N/A')})")
    print("")

## 3. Data Extraction
Load data from specified record set(s) into a pandas DataFrame for analysis. All entities are referenced by their `@id`.

In [ ]:
# For demonstration, extract all record sets into dataframes
# (If you know the specific @id you want, you can limit this to a subset)
record_set_ids = [rs['@id'] for rs in dataset.metadata_dict.get('recordSet', [])]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    if not dataframes[record_set_id].empty:
        print(f"Fields: {dataframes[record_set_id].columns.tolist()}")
        print(dataframes[record_set_id].head(2), "\n")

# Choose a main record set for EDA, e.g. the primary protein abundance table
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if main_record_set_id:
    print(f"Using record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Run basic EDA steps:
- Filter records based on a numeric field (e.g., abundance, peptide_count, or molecular_weight)
- Normalize values
- Group by another attribute

Remember to use field `@id`. Adjust field names to those shown in the previous overview.

In [ ]:
# Pick a numeric field's @id (replace as appropriate for your dataset)
import numpy as np

if main_record_set_id and not dataframes[main_record_set_id].empty:
    df = dataframes[main_record_set_id]

    print("Available columns:", df.columns.tolist())

    # Example: try to select a numeric field, fallback if not present
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in [np.float64, np.int64]]
    numeric_field = possible_numeric_fields[0] if possible_numeric_fields else df.columns[0]
    print(f"Selected numeric field for filtering: {numeric_field}")

    # Filtering step
    # Here, we use a basic quantile threshold for demonstration
    threshold = df[numeric_field].quantile(0.75) if np.issubdtype(df[numeric_field].dtype, np.number) else None
    if threshold is not None:
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold} ({len(filtered_df)} records):")
        print(filtered_df.head())
    else:
        filtered_df = df.copy()

    # Normalization step
    if np.issubdtype(df[numeric_field].dtype, np.number):
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a categorical field
    possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
    group_field = possible_group_fields[0] if possible_group_fields else None
    if group_field and group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nGrouped mean of {numeric_field} by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields, e.g. the distribution of normalized values or by categories.

Below is a histogram and, if possible, a boxplot grouped by a key field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and not filtered_df.empty:
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    # Histogram of numeric field
    sns.histplot(filtered_df[numeric_field], kde=True, ax=ax[0])
    ax[0].set_title(f"Histogram of {numeric_field}")
    # Boxplot by group
    if group_field and group_field in filtered_df.columns:
        sns.boxplot(x=group_field, y=numeric_field, data=filtered_df, ax=ax[1])
        ax[1].set_title(f"{numeric_field} by {group_field}")
        plt.setp(ax[1].xaxis.get_majorticklabels(), rotation=45)
    else:
        ax[1].axis('off')
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR² dataset using its Croissant schema with the `mlcroissant` library, explored available record sets and fields, and performed basic EDA and visualization using `pandas` and `seaborn`. This workflow can be extended to detailed analyses, e.g., protein abundance distributions, identification of sample-specific outliers, and further biological or bioinformatic hypothesis testing.

Remember to consult the dataset documentation for detailed field/coding semantics and proper interpretation. For additional Croissant schema datasets, adjust the record set and field references via their `@id`.